# Direction G: Token-by-Token Aesthetic Dynamics

**Goal**: Track how the aesthetic projection evolves as the model reads through a poem character by character.

**Method**: For each prefix length k (1 to len(text)), forward `text[:k]` → extract activations at multiple layers → project onto Exp 2's aesthetic direction.

**Hypothesis**: The aesthetic signal should spike at key imagery words (月, 花, 水) and build gradually, not appear all at once.

In [1]:
# §1 Setup
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
os.environ['GGML_CUDA_DISABLE_GRAPHS'] = '1'
os.environ['HF_HOME'] = '/workspace/Data/huggingface_cache'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = Path('/workspace/Data/direction_G')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
MODEL_PATH = '/workspace/Model/Qwen3-8B-GGUF/Qwen3-8B-Q4_K_M.gguf'

# Load aesthetic direction from Exp 2
exp2_data = np.load('/workspace/Data/aesthetic_experiment/aesthetic_analysis_results.npz', allow_pickle=True)
aes_dir_norm = exp2_data['aesthetic_direction_norm']  # (36, 4096)
exp2_best = int(exp2_data['best_cls_layer'])

# Also load Dir A direction if available
try:
    dirA_data = np.load('/workspace/Data/direction_A/dirA_results.npz', allow_pickle=True)
    dirA_dir_norm = dirA_data['aesthetic_dir_norm']
    dirA_best = int(dirA_data['best_layer'])
    print(f'Dir A direction loaded (best_layer=L{dirA_best})')
except:
    dirA_dir_norm = None
    dirA_best = None
    print('Dir A not available yet')

print(f'Exp 2 aesthetic direction: shape={aes_dir_norm.shape}, best_layer=L{exp2_best}')

Dir A direction loaded (best_layer=L4)
Exp 2 aesthetic direction: shape=(36, 4096), best_layer=L16


In [2]:
# §2 Load Engine
sys.path.insert(0, '/workspace/NeuroScope')
import neuroscope

engine = neuroscope.Engine(MODEL_PATH, n_ctx=4096, n_seq_max=1, n_gpu_layers=99)
info = engine.model_info
N_LAYERS, N_EMBD = info.n_layers, info.n_embd
print(f'Model: {info.name} | {N_LAYERS} layers | {N_EMBD} dim')

Model: Qwen3 8B Awq Compatible Instruct | 36 layers | 4096 dim


ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GB10, compute capability 12.1, VMM: yes
ggml_backend_cuda_get_available_uma_memory: final available_memory_kb: 77378200
ggml_backend_cuda_get_available_uma_memory: final available_memory_kb: 77378200
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GB10) (000f:01:00.0) - 75564 MiB free
llama_model_loader: loaded meta data with 28 key-value pairs and 399 tensors from /workspace/Model/Qwen3-8B-GGUF/Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 8B Awq Compatible Instruct
llama_model_loader: - kv   3:                           general.f

In [3]:
# §3 Select representative texts + annotate imagery positions
DATA_DIR = Path('/workspace/Data/poetry_data')
zh_poems = json.load(open(DATA_DIR / 'chinese_poems.json'))
en_poems = json.load(open(DATA_DIR / 'english_poems.json'))
news_data = json.load(open(DATA_DIR / 'wmt19_news.json'))

# Nature/sensory imagery characters
IMAGERY_CHARS = set('月花风雪春秋山水云霞星夜雨露竹松柳桃兰菊荷梅鸟蝶泉湖海虹烟雾光影梦香')

# Select Chinese poems with high imagery (for clear signal)
rng = np.random.RandomState(SEED)
zh_scores = []
for p in zh_poems:
    c = p['content']
    s = sum(1 for ch in c if ch in IMAGERY_CHARS) / max(len(c), 1)
    zh_scores.append(s)
zh_scores = np.array(zh_scores)

# Select 5 high-imagery poems (short, for visualization)
zh_valid = [(i, zh_scores[i]) for i in range(len(zh_poems))
            if 15 <= len(zh_poems[i]['content']) <= 60 and zh_scores[i] > 0.12]
zh_valid.sort(key=lambda x: -x[1])
zh_selected_idx = [x[0] for x in zh_valid[:5]]

# Select 3 low-imagery poems (control)
zh_low = [(i, zh_scores[i]) for i in range(len(zh_poems))
          if 15 <= len(zh_poems[i]['content']) <= 60 and zh_scores[i] < 0.02]
rng.shuffle(zh_low)
zh_low_idx = [x[0] for x in zh_low[:3]]

# Select 3 short English poems
en_selected = []
for p in en_poems:
    lines = p['content'].strip().split('\n')
    if 2 <= len(lines) <= 4 and 20 <= len(p['content']) <= 150:
        en_selected.append(p['content'].strip())
    if len(en_selected) >= 3:
        break

# Select 3 news texts (control)
news_selected = [news_data[i]['zh'][:50] for i in range(3)]

# Build experiment list
texts = []
text_meta = []
for idx in zh_selected_idx:
    p = zh_poems[idx]
    t = p['content'].replace('\n', '')
    texts.append(t)
    imagery_mask = [c in IMAGERY_CHARS for c in t]
    text_meta.append({'lang': 'ZH', 'type': 'high-imagery', 'title': p.get('title', ''),
                      'author': p.get('author', ''), 'imagery_mask': imagery_mask})

for idx in zh_low_idx:
    p = zh_poems[idx]
    t = p['content'].replace('\n', '')
    texts.append(t)
    imagery_mask = [c in IMAGERY_CHARS for c in t]
    text_meta.append({'lang': 'ZH', 'type': 'low-imagery', 'title': p.get('title', ''),
                      'author': p.get('author', ''), 'imagery_mask': imagery_mask})

for t in en_selected:
    texts.append(t)
    text_meta.append({'lang': 'EN', 'type': 'poetry', 'imagery_mask': None})

for t in news_selected:
    texts.append(t)
    text_meta.append({'lang': 'ZH', 'type': 'news', 'imagery_mask': [c in IMAGERY_CHARS for c in t]})

print(f'Selected {len(texts)} texts:')
for i, (t, m) in enumerate(zip(texts, text_meta)):
    print(f'  [{i}] {m["type"]:>12} ({m["lang"]}): {t[:50]}... ({len(t)} chars)')

Selected 11 texts:
  [0] high-imagery (ZH): 暮雨渔村春溟，晓霜枫叶秋酣。人世花开花落，山光湖北湖南。... (28 chars)
  [1] high-imagery (ZH): 黄菊香残夜雨，乌纱醉落秋风。回首十年旧事，乱云流水西东。... (28 chars)
  [2] high-imagery (ZH): 青山入湖湖水青，菱花白白照船棂。山头急雨船须住，水面凉风酒易醒。... (32 chars)
  [3] high-imagery (ZH): 霜风惊度雁，月露皓疏林。处处砧声发，星河秋夜深。... (24 chars)
  [4] high-imagery (ZH): 山月吟声苦，春风引思长。无由及尘土，犹带杏花香。... (24 chars)
  [5]  low-imagery (ZH): 重上青楼拂蛛网，却匀愁黛对菱波。也知新旧争多少，敢话机头织素多。... (32 chars)
  [6]  low-imagery (ZH): 五色怜凤雏，南飞适鹧鸪。楚人不相识，何处求椅梧。去去日千里，茫茫天一隅。安能与斥鷃，决起但枪榆。... (48 chars)
  [7]  low-imagery (ZH): 不把英雄彀，移为仕进媒。邑獒群怪吠，灶婢亦惊猜。且效功成退，宁须兴尽回。芸芸终有谢，赢得早归来。... (48 chars)
  [8]         news (ZH): 上周，古装剧《美人私房菜》临时停播，意外引发了关于国产剧收视率造假的热烈讨论。... (39 chars)
  [9]         news (ZH): 民权团体针对密苏里州发出旅行警告... (16 chars)
  [10]         news (ZH): 由于密苏里州的歧视性政策和种族主义袭击，美国有色人种促进协会 (NAACP) 向准备前往密苏里州出游... (50 chars)


In [4]:
# §4 Token-by-token activation collection
# For each text, forward progressively longer prefixes and collect activations

ANALYSIS_LAYERS = [0, 4, 8, 12, 16, 20, 24, 28, 32, 35]

def collect_positional_activations(engine, text, layers=ANALYSIS_LAYERS):
    """Forward text[:k] for k=1..len(text), collect last-token activations at specified layers."""
    n = len(text)
    results = {l: np.zeros((n, N_EMBD)) for l in layers}

    for k in range(1, n + 1):
        prefix = text[:k]
        engine.reset()
        ok = engine.forward(prefix, add_special=True)
        if not ok:
            break
        for l in layers:
            results[l][k-1] = engine.get_activations(l)

    return results

print('=== Collecting token-by-token activations ===')
all_results = []
t_start = time.time()
for i, text in enumerate(texts):
    t0 = time.time()
    res = collect_positional_activations(engine, text)
    dt = time.time() - t0
    all_results.append(res)
    print(f'  [{i}] {text_meta[i]["type"]:>12}: {len(text)} positions in {dt:.1f}s')

print(f'\nTotal: {time.time()-t_start:.1f}s')

=== Collecting token-by-token activations ===
  [0] high-imagery: 28 positions in 1.2s
  [1] high-imagery: 28 positions in 1.2s
  [2] high-imagery: 32 positions in 1.4s
  [3] high-imagery: 24 positions in 1.0s
  [4] high-imagery: 24 positions in 1.0s
  [5]  low-imagery: 32 positions in 1.4s
  [6]  low-imagery: 48 positions in 2.1s
  [7]  low-imagery: 48 positions in 2.1s
  [8]         news: 39 positions in 1.7s
  [9]         news: 16 positions in 0.6s
  [10]         news: 50 positions in 2.1s

Total: 15.8s


In [5]:
# §5 Plot individual poem trajectories (aesthetic projection over positions)
L = 16  # primary analysis layer

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle(f'Token-by-Token Aesthetic Projection (Layer {L})', fontsize=14, fontweight='bold')

# Plot high-imagery Chinese poems
for plot_i in range(min(5, len(zh_selected_idx))):
    ax = axes[plot_i // 2, plot_i % 2] if plot_i < 4 else axes[2, 0]
    text = texts[plot_i]
    meta = text_meta[plot_i]
    acts = all_results[plot_i][L]  # (n_chars, 4096)
    projections = acts @ aes_dir_norm[L]  # (n_chars,)

    chars = list(text)
    x = range(len(chars))

    # Color by imagery
    imagery = meta['imagery_mask']
    colors = ['crimson' if imagery[j] else 'steelblue' for j in range(len(chars))]

    ax.bar(x, projections, color=colors, alpha=0.7, width=0.8)
    ax.set_xticks(range(len(chars)))
    ax.set_xticklabels(chars, fontsize=7)
    ax.set_ylabel('Aesthetic projection')
    title = f'{meta.get("author", "")} 《{meta.get("title", "")}》'
    ax.set_title(title + ' (red=imagery)', fontsize=10)
    ax.axhline(0, ls='-', c='black', lw=0.3)

# Plot a news text for comparison
ax = axes[2, 1]
news_idx_in_list = len(zh_selected_idx) + len(zh_low_idx) + len(en_selected)  # first news
if news_idx_in_list < len(texts):
    text = texts[news_idx_in_list]
    meta = text_meta[news_idx_in_list]
    acts = all_results[news_idx_in_list][L]
    projections = acts @ aes_dir_norm[L]
    chars = list(text)
    imagery = meta['imagery_mask']
    colors = ['crimson' if imagery[j] else 'gray' for j in range(len(chars))]
    ax.bar(range(len(chars)), projections, color=colors, alpha=0.7, width=0.8)
    ax.set_xticks(range(len(chars)))
    ax.set_xticklabels(chars, fontsize=6)
    ax.set_ylabel('Aesthetic projection')
    ax.set_title('News (control)', fontsize=10)
    ax.axhline(0, ls='-', c='black', lw=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'trajectories_individual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved trajectories_individual.png')

Saved trajectories_individual.png


In [6]:
# §6 Aggregate analysis: imagery chars vs non-imagery chars
L = 16

# Collect projections from all ZH poems with imagery masks
imagery_projs = []
non_imagery_projs = []
for i in range(len(zh_selected_idx) + len(zh_low_idx)):
    text = texts[i]
    meta = text_meta[i]
    acts = all_results[i][L]
    projections = acts @ aes_dir_norm[L]
    imagery = meta['imagery_mask']
    for j in range(len(text)):
        if imagery[j]:
            imagery_projs.append(projections[j])
        elif text[j] not in '，。！？、；：""''《》（）':  # exclude punctuation
            non_imagery_projs.append(projections[j])

imagery_projs = np.array(imagery_projs)
non_imagery_projs = np.array(non_imagery_projs)

from scipy.stats import ttest_ind, mannwhitneyu
t_stat, p_val = ttest_ind(imagery_projs, non_imagery_projs)
u_stat, u_pval = mannwhitneyu(imagery_projs, non_imagery_projs, alternative='greater')
d_imagery = (imagery_projs.mean() - non_imagery_projs.mean()) / np.sqrt(
    (imagery_projs.std()**2 + non_imagery_projs.std()**2) / 2)

print(f'Imagery tokens:     N={len(imagery_projs)}, mean={imagery_projs.mean():.4f} ± {imagery_projs.std():.4f}')
print(f'Non-imagery tokens: N={len(non_imagery_projs)}, mean={non_imagery_projs.mean():.4f} ± {non_imagery_projs.std():.4f}')
print(f"Cohen's d: {d_imagery:.3f}")
print(f't-test: t={t_stat:.3f}, p={p_val:.4f}')
print(f'Mann-Whitney U (imagery > non): U={u_stat:.0f}, p={u_pval:.4f}')

# Also aggregate by poem type: high-imagery mean trajectory vs low-imagery mean
high_mean_projs = []
low_mean_projs = []
news_mean_projs = []

for i in range(len(zh_selected_idx)):
    acts = all_results[i][L]
    high_mean_projs.append((acts @ aes_dir_norm[L]).mean())
for i in range(len(zh_selected_idx), len(zh_selected_idx) + len(zh_low_idx)):
    acts = all_results[i][L]
    low_mean_projs.append((acts @ aes_dir_norm[L]).mean())
news_start = len(zh_selected_idx) + len(zh_low_idx) + len(en_selected)
for i in range(news_start, len(texts)):
    acts = all_results[i][L]
    news_mean_projs.append((acts @ aes_dir_norm[L]).mean())

print(f'\nMean projection per text type:')
print(f'  High-imagery poems: {np.mean(high_mean_projs):.4f}')
print(f'  Low-imagery poems:  {np.mean(low_mean_projs):.4f}')
print(f'  News texts:         {np.mean(news_mean_projs):.4f}')

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Aggregate Token-Level Analysis (Layer 16)', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(imagery_projs, bins=30, alpha=0.6, color='crimson', label=f'Imagery (N={len(imagery_projs)})', density=True)
ax.hist(non_imagery_projs, bins=30, alpha=0.6, color='steelblue', label=f'Non-imagery (N={len(non_imagery_projs)})', density=True)
ax.axvline(imagery_projs.mean(), ls='--', c='crimson')
ax.axvline(non_imagery_projs.mean(), ls='--', c='steelblue')
ax.set_xlabel('Aesthetic Projection'); ax.set_ylabel('Density')
ax.set_title(f"Imagery vs Non-imagery (d={d_imagery:.2f}, p={p_val:.3f})")
ax.legend(fontsize=8)

ax = axes[1]
categories = ['High-img\npoems', 'Low-img\npoems', 'News']
means = [np.mean(high_mean_projs), np.mean(low_mean_projs), np.mean(news_mean_projs)]
stds = [np.std(high_mean_projs), np.std(low_mean_projs), np.std(news_mean_projs)]
bars = ax.bar(categories, means, yerr=stds, color=['crimson', 'steelblue', 'gray'], alpha=0.7, capsize=5)
ax.set_ylabel('Mean Aesthetic Projection'); ax.set_title('By Text Type')

# Position effect: does projection increase with position?
ax = axes[2]
for i in range(min(5, len(zh_selected_idx))):
    acts = all_results[i][L]
    projs = acts @ aes_dir_norm[L]
    # Normalize position to 0-1
    x_norm = np.linspace(0, 1, len(projs))
    ax.plot(x_norm, projs, alpha=0.3, color='crimson')
for i in range(len(zh_selected_idx), len(zh_selected_idx) + len(zh_low_idx)):
    acts = all_results[i][L]
    projs = acts @ aes_dir_norm[L]
    x_norm = np.linspace(0, 1, len(projs))
    ax.plot(x_norm, projs, alpha=0.3, color='steelblue')
ax.set_xlabel('Relative position'); ax.set_ylabel('Aesthetic Projection')
ax.set_title('Trajectory overlay (red=high, blue=low)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'aggregate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved aggregate_analysis.png')

Imagery tokens:     N=38, mean=60.2079 ± 231.3148
Non-imagery tokens: N=186, mean=40.3880 ± 184.5070
Cohen's d: 0.095
t-test: t=0.574, p=0.5669
Mann-Whitney U (imagery > non): U=5223, p=0.0000

Mean projection per text type:
  High-imagery poems: 48.5515
  Low-imagery poems:  26.7018
  News texts:         51.6665
Saved aggregate_analysis.png


In [7]:
# §7 Multi-layer comparison — how does the token-level aesthetic signal evolve across layers?
# Use the first high-imagery poem as example
example_idx = 0
text = texts[example_idx]
meta = text_meta[example_idx]
chars = list(text)
imagery = meta['imagery_mask']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f'Layer-by-Layer Token Dynamics: {meta.get("author","")} 《{meta.get("title","")}》',
             fontsize=13, fontweight='bold')

for ax_i, l in enumerate(ANALYSIS_LAYERS):
    ax = axes[ax_i // 5, ax_i % 5]
    acts = all_results[example_idx][l]
    projs = acts @ aes_dir_norm[l]
    colors = ['crimson' if imagery[j] else 'steelblue' for j in range(len(chars))]
    ax.bar(range(len(chars)), projs, color=colors, alpha=0.7, width=0.8)
    ax.set_title(f'Layer {l}', fontsize=10)
    ax.set_xticks(range(len(chars)))
    ax.set_xticklabels(chars, fontsize=5, rotation=45)
    ax.axhline(0, ls='-', c='black', lw=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multi_layer_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute imagery vs non-imagery gap per layer
layer_gaps = []
for l in ANALYSIS_LAYERS:
    img_p = []
    non_p = []
    for i in range(len(zh_selected_idx) + len(zh_low_idx)):
        acts = all_results[i][l]
        projs = acts @ aes_dir_norm[l]
        mask = text_meta[i]['imagery_mask']
        for j in range(len(texts[i])):
            if mask[j]:
                img_p.append(projs[j])
            elif texts[i][j] not in '，。！？、；：""''《》（）':
                non_p.append(projs[j])
    img_p = np.array(img_p)
    non_p = np.array(non_p)
    gap = (img_p.mean() - non_p.mean()) / np.sqrt((img_p.std()**2 + non_p.std()**2)/2) if len(img_p) > 0 and len(non_p) > 0 else 0
    layer_gaps.append(gap)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ANALYSIS_LAYERS, layer_gaps, 'b-o', ms=5)
ax.set_xlabel('Layer'); ax.set_ylabel("Cohen's d (imagery vs non-imagery)")
ax.set_title('Imagery Token Effect Size by Layer')
ax.axhline(0, ls=':', c='gray')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'layer_gap_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Imagery-token effect size by layer:')
for l, g in zip(ANALYSIS_LAYERS, layer_gaps):
    print(f'  L{l:>2}: d={g:.3f}')
print(f'\nPeak imagery effect: L{ANALYSIS_LAYERS[np.argmax(np.abs(layer_gaps))]} (d={max(layer_gaps, key=abs):.3f})')
print('Saved multi_layer_dynamics.png, layer_gap_curve.png')

Imagery-token effect size by layer:
  L 0: d=0.229
  L 4: d=-0.231
  L 8: d=0.107
  L12: d=0.128
  L16: d=0.095
  L20: d=0.097
  L24: d=0.118
  L28: d=0.173
  L32: d=0.135
  L35: d=0.388

Peak imagery effect: L35 (d=0.388)
Saved multi_layer_dynamics.png, layer_gap_curve.png


In [8]:
# §8 Final summary + save
# Compute additional statistics
# 1. Autocorrelation of projection trajectory (does it drift or fluctuate?)
autocorrs = []
for i in range(len(zh_selected_idx)):
    acts = all_results[i][16]
    projs = acts @ aes_dir_norm[16]
    if len(projs) > 2:
        r = np.corrcoef(projs[:-1], projs[1:])[0, 1]
        autocorrs.append(r)

# 2. Cumulative build-up: does aesthetic projection grow with position?
slopes = []
for i in range(len(zh_selected_idx)):
    acts = all_results[i][16]
    projs = acts @ aes_dir_norm[16]
    if len(projs) > 3:
        x = np.arange(len(projs))
        slope = np.polyfit(x, projs, 1)[0]
        slopes.append(slope)

print('=== Direction G Summary ===')
print(f'\n1. Token-level imagery effect:')
print(f'   d={d_imagery:.3f}, p={p_val:.4f}')
if abs(d_imagery) > 0.2 and p_val < 0.05:
    print(f'   → Imagery tokens DO have higher aesthetic projection')
elif p_val >= 0.05:
    print(f'   → No significant difference at token level')
else:
    print(f'   → Small effect')

print(f'\n2. Trajectory autocorrelation (L16):')
print(f'   mean r = {np.mean(autocorrs):.3f} ± {np.std(autocorrs):.3f}')
if np.mean(autocorrs) > 0.5:
    print(f'   → Strong sequential dependency (smooth trajectory)')
else:
    print(f'   → Weak autocorrelation (fluctuating signal)')

print(f'\n3. Position trend (slope):')
print(f'   mean slope = {np.mean(slopes):.4f} ± {np.std(slopes):.4f}')
if abs(np.mean(slopes)) > 0.01:
    print(f'   → Aesthetic projection {"increases" if np.mean(slopes)>0 else "decreases"} with position')
else:
    print(f'   → No significant positional trend')

print(f'\n4. Peak layer for imagery effect:')
peak_l_idx = np.argmax(np.abs(layer_gaps))
print(f'   L{ANALYSIS_LAYERS[peak_l_idx]} (d={layer_gaps[peak_l_idx]:.3f})')

# Save
np.savez(OUTPUT_DIR / 'dirG_results.npz',
    d_imagery=d_imagery, p_value=p_val,
    autocorrs=np.array(autocorrs), slopes=np.array(slopes),
    layer_gaps=np.array(layer_gaps), analysis_layers=np.array(ANALYSIS_LAYERS),
    n_texts=len(texts))

print(f'\nResults saved to {OUTPUT_DIR / "dirG_results.npz"}')
print(f'Plots saved to {OUTPUT_DIR}')

=== Direction G Summary ===

1. Token-level imagery effect:
   d=0.095, p=0.5669
   → No significant difference at token level

2. Trajectory autocorrelation (L16):
   mean r = 0.324 ± 0.247
   → Weak autocorrelation (fluctuating signal)

3. Position trend (slope):
   mean slope = -9.2361 ± 1.7927
   → Aesthetic projection decreases with position

4. Peak layer for imagery effect:
   L35 (d=0.388)

Results saved to /workspace/Data/direction_G/dirG_results.npz
Plots saved to /workspace/Data/direction_G
